# He Ion Fluence → Irradiation Time

Fluence rate through the aperture:

$$\frac{d\Phi}{dt} = \frac{I_{beam}}{q} \cdot \frac{f_{transmitted}}{A_{aperture}}$$

where $f_{transmitted}$ is the fraction of the (Gaussian) beam passing through the aperture:

$$f_{transmitted} = \int_{-r_{ap}}^{r_{ap}} I_0\, e^{-x^2/2\sigma^2}\,dx = \mathrm{erf}\!\left(\frac{r_{ap}}{\sigma\sqrt{2}}\right)$$

Solve for time to reach target fluence $\Phi_{target}$:

$$t = \frac{\Phi_{target}}{d\Phi/dt}$$

In [1]:
import numpy as np
from scipy.special import erf
from scipy.constants import e as e_charge  # 1.602176634e-19 C

## Inputs

In [2]:
# --- Beam ---
I_beam_nA   = 100.0        # total beam current, nA (read off accelerator readout)
charge_state = 2           # He2+ (alpha) = 2, He+ = 1

# --- Aperture ---
r_ap_cm     = 0.05         # aperture radius, cm (set with aperture chosen)
sigma_beam_cm = 0.1        # Gaussian beam sigma at aperture plane, cm (from beam profile/pinhole scan)

# --- Target ---
fluence_target = 1e14      # He ions / cm^2

# --- Derived constants ---
I_beam = I_beam_nA * 1e-9       # A = C/s
q_ion  = charge_state * e_charge  # C per He ion
A_aperture = np.pi * r_ap_cm**2   # cm^2

## Fraction of beam transmitted through aperture

In [3]:
f_transmitted = erf(r_ap_cm / (sigma_beam_cm * np.sqrt(2)))
print(f"f_transmitted = {f_transmitted:.4f}")

f_transmitted = 0.3829


## Fluence rate and irradiation time

In [4]:
ions_per_sec = I_beam / q_ion              # total He ions/s in beam
dPhi_dt = ions_per_sec * f_transmitted / A_aperture  # ions/(cm^2 s) on sample

t_sec = fluence_target / dPhi_dt

print(f"Ion current:        {ions_per_sec:.3e} ions/s")
print(f"Fluence rate:        {dPhi_dt:.3e} ions/(cm^2 s)")
print(f"Time required:       {t_sec:.3e} s")
print(f"                     {t_sec/60:.3e} min")
print(f"                     {t_sec/3600:.3e} hr")

Ion current:        3.121e+11 ions/s
Fluence rate:        1.522e+13 ions/(cm^2 s)
Time required:       6.572e+00 s
                     1.095e-01 min
                     1.826e-03 hr


## Notes
- `charge_state`: 2 for He2+ (alpha), 1 for He+. Confirm accelerator's ion source output.
- `sigma_beam_cm`: needs an independent beam profile measurement (e.g. pinhole/Faraday cup scan)
- If we have a directly measured `f_transmitted` (rather than fitting a Gaussian), just hardcode it and skip the `erf` cell.
- Sanity check: `A_aperture` should be << beam spot size for the Gaussian truncation to matter; if `r_ap_cm >> sigma_beam_cm`, `f_transmitted → 1`.